# 03 — Recurrence Quantification Analysis

**Purpose:** Compute RQA measures for every segmented trial and produce two core outputs:

1. **Recurrence plots** for visual inspection of gaze dynamics.
2. **A summary DataFrame** (one row per trial) with descriptive stats, autocorrelation, and RQA measures — ready for export and statistical modelling.

## Analysis modes

- **Fixed-epsilon:** A single radius (in z-score units) is applied to all trials.
- **RR-locked:** The radius is chosen per trial so that the recurrence rate matches a target value.

## Pipeline

1. Point `TRIALS_DIR` at the folder of segmented trial CSVs (output of Notebook 02).
2. Choose analysis mode and parameters in the config cell.
3. Run the summary builder.
4. Browse recurrence plots with the interactive viewer.
5. Save the summary CSV.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd

from neon_gaze.rqa import build_rqa_summary
from neon_gaze.plotting import plot_recurrence_matrix

## Configuration

Set `DEMO = True` to run with the small sample data shipped with this repo.  
Set `DEMO = False` (the default) to use your own segmented trial CSVs.


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
# If you are using this repo for the first time, set DEMO = True to
# run the pipeline on the included sample data.
DEMO = False

if DEMO:
    TRIALS_DIR = "../demo/walk_segmented_csvs"
    OUTPUT_DIR = "../demo/output"
else:
    TRIALS_DIR = "../walk_segmented_csvs"
    OUTPUT_DIR = "../data/output"

# ── Analysis mode: choose ONE ──────────────────────────────────────
#
# Option A — Fixed epsilon (radius in z-score units):
EPSILON = 0.07
TARGET_RR = None
#
# Option B — RR-locked (uncomment below, comment out Option A):
# EPSILON = None
# TARGET_RR = 0.05

# Minimum diagonal/vertical line length for DET, MaxL, ENT, LAM, TT
L_MIN = 5

SAVE_OUTPUT = False


## Build the RQA summary DataFrame

This loops over every trial CSV in `TRIALS_DIR`, z-scores the gaze-angle series, computes autocorrelation and RQA, and collects everything into a single DataFrame.

In [ ]:
summary_df = build_rqa_summary(
    base_dir=TRIALS_DIR,
    epsilon=EPSILON,
    target_rr=TARGET_RR,
    l_min=L_MIN,
)

print(f"Processed {len(summary_df)} trials.")
display(summary_df.head(10))

In [ ]:
display(summary_df.describe())

## Recurrence plot viewer

Use the dropdown to select a trial and the slider to adjust epsilon. The recurrence plot is rendered as a square matrix where dark pixels indicate recurrent pairs.

In [ ]:
from ipywidgets import Dropdown, FloatSlider, HBox, VBox
from IPython.display import display, clear_output

csv_files = sorted(
    f for f in os.listdir(TRIALS_DIR) if f.lower().endswith(".csv")
) if os.path.isdir(TRIALS_DIR) else []

if csv_files:
    file_dropdown = Dropdown(
        options=csv_files,
        value=csv_files[0],
        description="Trial:",
        style={"description_width": "initial"},
        layout={"width": "500px"},
    )
    epsilon_slider = FloatSlider(
        value=EPSILON if EPSILON is not None else 0.07,
        min=0.02, max=0.50, step=0.01,
        description="epsilon (z):",
        style={"description_width": "initial"},
        readout_format=".2f",
        layout={"width": "400px"},
    )
    controls = HBox([file_dropdown, epsilon_slider])

    def update_plot(*_):
        fname = file_dropdown.value
        eps = float(epsilon_slider.value)
        clear_output(wait=True)
        display(VBox([controls]))
        df = pd.read_csv(os.path.join(TRIALS_DIR, fname))
        gaze = df["gaze angle [deg]"].to_numpy(dtype=float)
        plot_recurrence_matrix(gaze, epsilon=eps, title=fname)

    file_dropdown.observe(update_plot, names="value")
    epsilon_slider.observe(update_plot, names="value")
    display(VBox([controls]))
    update_plot()
else:
    print(f"No trial CSVs found in '{TRIALS_DIR}'. Run Notebook 02 first.")

## Save the summary DataFrame

In [ ]:
if SAVE_OUTPUT:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if TARGET_RR is not None:
        fname = f"rqa_summary_rr-locked-{TARGET_RR}_lmin-{L_MIN}.csv"
    else:
        fname = f"rqa_summary_eps-{EPSILON}_lmin-{L_MIN}.csv"
    out_path = os.path.join(OUTPUT_DIR, fname)
    summary_df.to_csv(out_path, index=False)
    print(f"Saved {len(summary_df)} rows to {out_path}")
else:
    print("SAVE_OUTPUT is False — set to True in the config cell to save.")